<h1 style="text-align:center; font-size:40px;"> Team 6 - TASK - 9 </h1>

In [67]:
## Importing Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from matplotlib.patches import Patch
import matplotlib.patches as mpatches
from scipy.stats import chi2_contingency, f_oneway
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA

## Importing the files (features and target) from https://archive.ics.uci.edu/dataset/2/adult

In [70]:
pip install ucimlrepo

Note: you may need to restart the kernel to use updated packages.


In [71]:
from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
df_adult = fetch_ucirepo(id=2) 
  
# data (as pandas dataframes) 
X = df_adult.data.features 
Y = df_adult.data.targets 


In [73]:
# Merging features and target table(X and Y)

df_adult = X.copy()
df_adult["income"] = Y.iloc[:, 0] 

In [74]:
df_adult.head(5)

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


In [78]:
# Removing the trailing dot for income column
df_adult["income"] = df_adult["income"].str.strip().str.replace(".", "", regex=False)
df_adult["income"].value_counts()

income
<=50K    37155
>50K     11687
Name: count, dtype: int64

## Occupation, Workclass, Native-Country feature:

In [81]:
m=df_adult['workclass'].mode()
print(m)
o=df_adult['occupation'].mode()
print(o)
n=df_adult['native-country'].mode()
print(n)

0    Private
Name: workclass, dtype: object
0    Prof-specialty
Name: occupation, dtype: object
0    United-States
Name: native-country, dtype: object


In [83]:
# filling the values replace  ‘?’ is ‘Workclass’ column by ‘Private’, ‘Occupation’ column by ‘Prof-speciality’ 
#and ‘Native_country’by ‘United_States’.
df_adult['workclass']=df_adult['workclass'].replace('?','Private')
df_adult['occupation']=df_adult['occupation'].replace('?','Prof-specialty')
df_adult['native-country']=df_adult['native-country'].replace('?',' United-States')

## Dropping features based on Variate analysis

In [86]:
# Drop based on our analysis findings
df_adult.drop(columns=["fnlwgt", "native-country", "race"], inplace=True)

## Calculating net capital value

In [90]:
# Create net capital = gain - loss
df_adult["net_capital"] = df_adult["capital-gain"] - df_adult["capital-loss"]

## Encode Target Variable

In [95]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()

# Binary encode income (<=50K = 0, >50K = 1)
df_adult["income"] = le.fit_transform(df_adult["income"])

# Binary encode sex (Male = 1, Female = 0)
df_adult["sex"] = le.fit_transform(df_adult["sex"])


## Skewness Value 

In [98]:
numeric_data = df_adult.select_dtypes(include='number')

# Now compute skewness
skew_values = numeric_data.skew()

print(skew_values)

age                0.557580
education-num     -0.316525
sex               -0.715811
capital-gain      11.894659
capital-loss       4.569809
hours-per-week     0.238750
income             1.222216
net_capital       11.814939
dtype: float64


## Log Transform Skewed Features

In [103]:
#  Capital Feature Engineering 

# Create net capital = gain - loss
df_adult["net_capital"] = df_adult["capital-gain"] - df_adult["capital-loss"]

# Log transform net capital — clip negatives to 0 before applying log
df_adult["log_net_capital"] = np.log1p(df_adult["net_capital"].clip(lower=0))

# Log transform capital-gain/loss — reduces skewness
#df_adult["capital_gain_log"] = np.log1p(df_adult["capital-gain"])
#df_adult["capital_loss_log"] = np.log1p(df_adult["capital-loss"])

# Capital net — combine capital gain and loss
#df_adult["capital_net"] = df_adult["capital_gain_log"] - df_adult["capital_loss_log"]

# Drop original columns — replaced by log transformed versions
df_adult.drop(columns=["capital-gain", "capital-loss", "net_capital"], inplace=True)

## Create New Features

In [105]:
# Age group — based on your pair plot findings
df_adult["age_group"] = pd.cut(df_adult["age"],
                          bins=[0, 30, 45, 60, 100],
                          labels=["Young", "Middle", "Senior", "Elderly"])

In [107]:
#Education Group
df_adult['education_group'] = pd.cut(df_adult['education-num'],
    bins=[0, 9, 13, 16], labels=['Low Education', 'Undergrad Level', 'Graduate Level'])

In [109]:
# define mapping dictionaries--workclass
workclass_group= {'Never-worked' :'Without-pay', 'Without-pay' : 'Without-pay',
                  'Local-gov':'gov','State-gov':'gov','Federal-gov' : 'Federal-gov',
                  'Self-emp-not-inc' : 'Private', 'Private':'Private',
                 'Self-emp-inc' : 'Self-emp-inc'}
#apply replacements
df_adult['workclass']=df_adult['workclass'].map(workclass_group).fillna('Other')

In [111]:
# define mapping dictionaries--maritalstatus_group

maritalstatus_group={'Divorced' :'No-spouse', 'Separated':'No-spouse','Widowed':'No-spouse',
                     'Married-spouse-absent':'No-spouse','Married-AF-spouse':'No-spouse',
                     'Married-civ-spouse':'Married-civ-spouse', 'Never-married':'Never-married'}
#apply replacements
df_adult['marital-status']=df_adult['marital-status'].map(maritalstatus_group).fillna('Other')

In [113]:
relationship_map = {
    # Married
    "Husband"        : "Married",
    "Wife"           : "Married",
    # Family
    "Own-child"      : "Family",
    "Other-relative" : "Family",
    # Alone
    "Not-in-family"  : "Alone",
    "Unmarried"      : "Alone"
}

# Apply mapping
df_adult["relationship"] = df_adult["relationship"].map(relationship_map)


In [115]:
# define mapping dictionaries-- occupation_group

# Group 15 occupation categories into 5 meaningful groups
occupation_map = {
    # White Collar — professional/office jobs
    "Exec-managerial"  : "White_Collar",
    "Prof-specialty"   : "White_Collar",
    "Adm-clerical"     : "White_Collar",
    "Tech-support"     : "White_Collar",
    # Blue Collar — manual/trade jobs
    "Craft-repair"     : "Blue_Collar",
    "Machine-op-inspct": "Blue_Collar",
    "Transport-moving" : "Blue_Collar",
    "Handlers-cleaners": "Blue_Collar",
    "Farming-fishing"  : "Blue_Collar",
    # Service — customer/service jobs
    "Other-service"    : "Service",
    "Protective-serv"  : "Service",
    "Priv-house-serv"  : "Service",
    # Sales
    "Sales"            : "Sales",
    # Armed Forces
    "Armed-Forces"     : "Armed_Forces"
}

# Apply mapping
df_adult["occupation"] = df_adult["occupation"].map(occupation_map)


## Compouting skewness

In [118]:
numeric_data = df_adult.select_dtypes(include='number')

# Now compute skewness
skew_values = numeric_data.skew()

print(skew_values)

age                0.557580
education-num     -0.316525
sex               -0.715811
hours-per-week     0.238750
income             1.222216
log_net_capital    3.113630
dtype: float64



## Dropping raw columns

In [121]:
df_adult.drop(columns=["age", "education-num", "education"], inplace=True)

## One-Hot Encode Categorical Columns

In [125]:
# One-hot encode nominal categorical columns
df_adult = pd.get_dummies(df_adult, columns=["workclass", "education_group", "marital-status", 
                                             "occupation", "relationship", "age_group"], drop_first=True)

In [127]:
df_adult.columns

Index(['sex', 'hours-per-week', 'income', 'log_net_capital', 'workclass_Other',
       'workclass_Private', 'workclass_Self-emp-inc', 'workclass_Without-pay',
       'workclass_gov', 'education_group_Undergrad Level',
       'education_group_Graduate Level', 'marital-status_Never-married',
       'marital-status_No-spouse', 'occupation_Blue_Collar',
       'occupation_Sales', 'occupation_Service', 'occupation_White_Collar',
       'relationship_Family', 'relationship_Married', 'age_group_Middle',
       'age_group_Senior', 'age_group_Elderly'],
      dtype='object')

In [130]:
df_adult.head(5)

,sex,hours-per-week,income,log_net_capital,workclass_Other,workclass_Private,workclass_Self-emp-inc,workclass_Without-pay,workclass_gov,education_group_Undergrad Level,...,marital-status_No-spouse,occupation_Blue_Collar,occupation_Sales,occupation_Service,occupation_White_Collar,relationship_Family,relationship_Married,age_group_Middle,age_group_Senior,age_group_Elderly
0,1,40,0,7.684784,False,False,False,False,True,True,...,False,False,False,False,True,False,False,True,False,False
1,1,13,0,0.000000,False,True,False,False,False,True,...,False,False,False,False,True,False,True,False,True,False
2,1,40,0,0.000000,False,True,False,False,False,False,...,True,True,False,False,False,False,False,True,False,False
3,1,40,0,0.000000,False,True,False,False,False,False,...,False,True,False,False,False,False,True,False,True,False
4,0,40,0,0.000000,False,True,False,False,False,True,...,False,False,False,False,True,False,True,False,False,False


In [132]:
# Convert all bool columns to int
bool_cols = df_adult.select_dtypes(include='bool').columns
df_adult[bool_cols] = df_adult[bool_cols].astype(int)
df_adult.head(5)

,sex,hours-per-week,income,log_net_capital,workclass_Other,workclass_Private,workclass_Self-emp-inc,workclass_Without-pay,workclass_gov,education_group_Undergrad Level,...,marital-status_No-spouse,occupation_Blue_Collar,occupation_Sales,occupation_Service,occupation_White_Collar,relationship_Family,relationship_Married,age_group_Middle,age_group_Senior,age_group_Elderly
0,1,40,0,7.684784,0,0,0,0,1,1,...,0,0,0,0,1,0,0,1,0,0
1,1,13,0,0.000000,0,1,0,0,0,1,...,0,0,0,0,1,0,1,0,1,0
2,1,40,0,0.000000,0,1,0,0,0,0,...,1,1,0,0,0,0,0,1,0,0
3,1,40,0,0.000000,0,1,0,0,0,0,...,0,1,0,0,0,0,1,0,1,0
4,0,40,0,0.000000,0,1,0,0,0,1,...,0,0,0,0,1,0,1,0,0,0
